Tokenizing steps
1]Splitting text into individual word and subwords
2]Convert tokens into ID's
3]Encode token ID's into vector representations.

In [ ]:
with open("/content/the-verdict.txt","r", encoding="utf-8") as f:
  raw_text = f.read()
print("Total number of character:",len(raw_text))
print(raw_text[:99])

In [ ]:
import re
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)',raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:20])

In [ ]:
print(len(preprocessed))

Converting the Tokens into ID'S
tokens*tokenized text *Vocabulary

In [ ]:
all_words =sorted(set(preprocessed))
vocab_size =len(all_words)

In [ ]:
print(vocab_size)

In [ ]:
vocab = {token :integer for integer,token in enumerate(all_words)}

enumerate =assigns the integer to words


In [ ]:
for i,item in enumerate(vocab.items()):
  print(item)
  if i >= 50:
    break

Decoder DEvopment
1.Store the vocab as class attribute for access in the enocode and decode menthods
2.create an inveerse vocabulary that maps token ID's vback to original text tokens

3.process ip text into token id's
4.Convert token id's back into txet
5.Replace spaces before the specifiied punctuation

In [ ]:
class SimpleTokenizerV1:
  def __init__(self, vocab):
    self.str_to_int = vocab
    self.int_to_str = {i:s for s,i in vocab.items()}

  def encode(self, text):
    preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
    preprocessed = [
        item.strip() for item in preprocessed if item.strip()
    ]
    ids = [self.str_to_int[s] for s in preprocessed]
    return ids

  def decode(self, ids):
    text = " ".join([self.int_to_str[i] for i in ids])
    # Replace Spaces before the specified punctuations
    text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
    return text

In [ ]:
tokenizer = SimpleTokenizerV1(vocab)

text = """""You ever knew? But you just said--"""
ids=tokenizer.encode(text)
print(ids)


In [ ]:
tokenizer.decode(ids)

In [ ]:
all_tokens =sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>","<|unk|>"])
vocab={token:integer for integer,token in enumerate(all_tokens)}

In [ ]:
len(vocab.items())

In [ ]:
for i ,item in enumerate(list(vocab.items())[-5:]):
  print(item)

In [ ]:
class SimpleTokenizerV2:
  def __init__(self,vocab):
    self.str_to_int=vocab
    self.int_to_str={i:s for s,i in vocab.items()}

  def encode(self,text):
    preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
    preprocessed = [item.strip() for item in preprocessed if item.strip()]
    preprocessed = [
        item if item in self.str_to_int
        else "<|unk|>" for item in preprocessed
    ]

    ids=[self.str_to_int[s] for s in preprocessed]
    return ids

  def decode(self,ids):
    text=" ".join([self.int_to_str[i] for i in ids])
    text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
    return text

In [ ]:
tokenizer =SimpleTokenizerV2(vocab)
text1 ="Hello,do you like tea?"
text2 = "Hello,do u like cake!?"
text ="<|endoftext|> ".join((text1,text2))
print(text)

In [ ]:
tokenizer.encode(text)

Adding special Contexxt Tokens


In [ ]:
tokenizer.decode(tokenizer.encode(text))

Bytepair encoding

In [ ]:
!pip3 install tiktoken

In [ ]:
import importlib
import tiktoken

print("tiktiken  version:", importlib.metadata.version("tiktoken"))

In [ ]:
tokenizer=tiktoken.get_encoding("gpt2")

In [ ]:
text=(
    "Hello,do u like tea? <|endoftext|> In the sunnlight teraces"
    "of someknowledgrplace."
)
integers =tokenizer.encode(text,allowed_special={"<|endoftext|>"})
print(integers)

In [ ]:
mm =tokenizer.decode(integers)
print(mm)

In [ ]:
integers = tokenizer.encode("Akwiw ier")
print(integers)

mm=tokenizer.decode(integers)
print(mm)

CREATING INPUT-PAIRS

In [ ]:
with open("/content/the-verdict.txt","r",encoding="utf-8") as f:
  raw_text=f.read()
  enc_text = tokenizer.encode(raw_text)
  print(len(enc_text))

In [ ]:
enc_sample=enc_text[40:]

Context size

In [ ]:
context_size =4 #length of the input
x=enc_sample[:context_size]
y=enc_sample[1:context_size+1]

print(f"x:  {x}")
print(f"y:       {y}")

In [ ]:
for i in range(1, context_size + 1):
  context = enc_sample[:i]
  desired = enc_sample[i]

  print(context, "--->", desired)

In [ ]:
for i in range(1,context_size + 1):
  context = enc_sample[:i]
  desired = enc_sample[i]

  print(tokenizer.decode(context),"---->", tokenizer.decode([desired]))

Implimenting data loader


S1:Tokenize the entire text

s2:use sliding window to chunk the bookk into overlaping sequences of max_length

3:Return the total no of rows in the dataset

4:return a single raw from the dataset

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch

class GPTDatasetV1(Dataset):
  def __init__(self, text, tokenizer, max_length, stride):
    self.input_ids = []
    self.target_ids = []

    token_ids = tokenizer.encode(text, allowed_special={"<|endoftext| thickness>"})

    for i in range(0, len(token_ids) - max_length, stride):
      input_chunk = token_ids[i:i + max_length]
      target_chunk = token_ids[i + 1:i + max_length + 1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, idx):
    return self.input_ids[idx], self.target_ids[idx]

In [ ]:
import tiktoken

def create_data_loader_V1(txt, batch_size=4, max_length=256,
                          stride=128, shuffle=True, drop_last=True,
                          num_workers=0):

  tokenizer = tiktoken.get_encoding("gpt2")

  dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

  dataloader = DataLoader(
      dataset,
      batch_size=batch_size,
      shuffle=shuffle,
      drop_last=drop_last,
      num_workers=num_workers
  )
  return dataloader

In [ ]:
with open("/content/the-verdict.txt","r",encoding="utf-8") as f:
  raw_text=f.read()

In [ ]:
import torch
print("Pytorch version:", torch.__version__)
dataloader = create_data_loader_V1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

In [ ]:
second_batch = next(data_iter)
print(second_batch)

In [ ]:
third_batch=next(data_iter)
print(third_batch)

In [ ]:
dataloader = create_data_loader_V1(raw_text, batch_size=10, max_length=6, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("inputs:\n", inputs)
print("\nTargets:\n", targets)

# New Section